In [ ]:
import numpy as np
import random

# 1. Grid setup
N = 8
START = (0, 0)
GOAL = (7, 7)

OBSTACLES = [
    (1, 3), (2, 3), (3, 3), (4, 3),
    (5, 3), (5, 4), (5, 5)
]

# 2. Get parameters from user
ANTS = int(input("Enter number of ants: "))
ITERATIONS = int(input("Enter number of iterations: "))
EVAPORATION = float(input("Enter evaporation rate (0 to 1): "))

# Initialize pheromone levels
pheromone = np.ones((N, N, N, N))

# 3. Find valid neighboring cells
def get_neighbors(pos, visited):
    r, c = pos
    neighbors = []

    for dr, dc in [(0,1), (1,0), (0,-1), (-1,0)]:
        nr, nc = r + dr, c + dc

        if (0 <= nr < N and 0 <= nc < N
            and (nr,nc) not in OBSTACLES
            and (nr,nc) not in visited):
            neighbors.append((nr,nc))

    return neighbors


# 4. Construct path using pheromones
def construct_path():
    current = START
    path = [START]
    visited = {START}

    while current != GOAL:
        neighbors = get_neighbors(current, visited)

        if not neighbors:
            return None

        weights = []

        for r, c in neighbors:
            tau = pheromone[current[0], current[1], r, c]
            distance = abs(r-GOAL[0]) + abs(c-GOAL[1])
            heuristic = 1 / (distance + 1)

            weights.append(tau * heuristic**2)

        next_cell = random.choices(
            neighbors, weights=weights
        )[0]

        path.append(next_cell)
        visited.add(next_cell)
        current = next_cell

    return path


# 5. Pheromone evaporation and update
def update_pheromone(paths):
    global pheromone

    pheromone *= (1 - EVAPORATION)

    for path in paths:
        deposit = 100 / (len(path) - 1)

        for i in range(len(path)-1):
            r1, c1 = path[i]
            r2, c2 = path[i+1]

            pheromone[r1,c1,r2,c2] += deposit
            pheromone[r2,c2,r1,c1] += deposit


# 6. Main ACO algorithm
best_path = None

for iteration in range(ITERATIONS):
    paths = []

    for ant in range(ANTS):
        path = construct_path()

        if path is not None:
            paths.append(path)

    if paths:
        current_best = min(paths, key=len)

        if best_path is None or len(current_best) < len(best_path):
            best_path = current_best

        update_pheromone(paths)

    if iteration == 0 or iteration == ITERATIONS-1:
        print("Iteration", iteration+1,
              "- Best path length:",
              len(best_path)-1 if best_path else "Not found")


# 7. Display final result
if best_path:
    print("\nBest Path:", best_path)
    print("Path Length:", len(best_path)-1)
    print("Average Pheromone on Best Path:",
          round(np.mean([
              pheromone[a[0],a[1],b[0],b[1]]
              for a,b in zip(best_path,best_path[1:])
          ]), 2))

    # Display grid
    grid = [['.' for _ in range(N)] for _ in range(N)]

    for r,c in OBSTACLES:
        grid[r][c] = '#'

    for r,c in best_path:
        if (r,c) != START and (r,c) != GOAL:
            grid[r][c] = '*'

    grid[0][0] = 'S'
    grid[7][7] = 'G'

    print("\nFinal Robot Path:")
    for row in grid:
        print(' '.join(row))

else:
    print("\nNo valid path found!")


Enter number of ants: 20
Enter number of iterations: 50
Enter evaporation rate (0 to 1): 0.5
Iteration 1 - Best path length: 14
Iteration 50 - Best path length: 14

Best Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (1, 7), (2, 7), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7)]
Path Length: 14
Average Pheromone on Best Path: 184.66

Final Robot Path:
S * * * * * * *
. . . # . . . *
. . . # . . . *
. . . # . . . *
. . . # . . . *
. . . # # # . *
. . . . . . . *
. . . . . . . G
